# PartnerCopilot — rodada de geração
Selecione um runtime com GPU antes de executar. O notebook clona o repositório público,
instala o projeto e grava uma linha JSONL por pergunta diretamente no Drive.
Adicione HF_TOKEN aos Secrets do Colab e habilite o acesso deste notebook. Não escreva o token em células.
Os testes de CPU e a preparação do notebook não são resultados do Mistral.


In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/partner-copilot')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/JoaoNordemann/partner-copilot.git',str(repo)],check=True)
os.chdir(repo)
# Não faça git pull automaticamente ao retomar: a identidade do experimento inclui o código.
subprocess.run([sys.executable,'-m','pip','install','-e','.'],check=True)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'Configure o Secret HF_TOKEN sem exibi-lo.'
import torch
assert torch.cuda.is_available(), 'Ative um runtime GPU no Colab.'
output = Path('/content/drive/MyDrive/PartnerCopilot/generation-v1.jsonl')
output.parent.mkdir(parents=True,exist_ok=True)


In [ ]:
subprocess.run([sys.executable,'scripts/evaluate_generation.py','--check-only','--output',str(output)],check=True)


A execução abaixo usa o pipeline real e pode baixar pesos para o cache do runtime.
A saída é gravada com flush/fsync após cada pergunta. Reexecutar a célula retoma sem duplicar IDs.
Erros são preservados como tentativas; mudança de versão, configuração ou GPU exige outro arquivo.


In [ ]:
subprocess.run([sys.executable,'scripts/evaluate_generation.py','--resume','--output',str(output)],check=True)


Mesmo se a execução parar, o JSONL já contém as perguntas concluídas. A célula abaixo
resume as linhas salvas; taxas ficam pendentes enquanto a rodada estiver incompleta.
Leia o arquivo .review.md: ele reúne dez perguntas fixadas previamente, incluindo todas as negativas.
Marque a revisão somente depois de ler as respostas e os trechos. Não confunda template de revisão com leitura concluída.


In [ ]:
subprocess.run([sys.executable,'scripts/evaluate_generation.py','--summarize-only','--output',str(output)],check=True)
print(output.with_suffix('.review.md').read_text(encoding='utf-8'))
